# Fine-Tuned Model Evaluation & Gradio Deployment

## Regenerated Logical + Better UI Version

This notebook is a corrected follow-up notebook after LoRA / QLoRA fine-tuning.

It is designed for students who already trained and saved a LoRA / QLoRA adapter in the previous notebook.

## Main Goal

```text
Load adapter → Compare base vs fine-tuned model → Evaluate safely → Export results → Deploy with Gradio
```

## What is fixed in this version?

```text
1. Safer adapter ZIP upload and extraction
2. Correct adapter folder validation
3. Deterministic generation for fair evaluation
4. Numeric evaluation columns to avoid Pandas dtype errors
5. Better display for long model answers
6. Automatic score calculation
7. Improved Gradio UI with tabs
8. Side-by-side base vs fine-tuned comparison in UI
9. Cleaner classroom flow
```

## v2 Fix

This version fixes the Colab `SameFileError` that occurs when `lora_qlora_adapter.zip` is already saved at `/content/lora_qlora_adapter.zip`.


## v3 Fix

This version sets `USE_4BIT = False` by default for stable evaluation with the 0.5B model and adds automatic fallback if `bitsandbytes` 4-bit loading fails.


# 1. Learning Outcomes

By the end of this notebook, students will be able to:

1. Load a saved LoRA / QLoRA adapter.
2. Understand the difference between base model and adapter model.
3. Generate deterministic responses for evaluation.
4. Compare base model and fine-tuned model answers.
5. Create a safe manual evaluation table.
6. Score accuracy, relevance, clarity, completeness, hallucination risk, and safety.
7. Export comparison and evaluation reports.
8. Launch a professional Gradio UI for testing the fine-tuned model.

# 2. Runtime Setup

Recommended runtime:

```text
Google Colab → Runtime → Change runtime type → GPU
```

Then run the next cell.

In [ ]:
!nvidia-smi

Sat Aug 15 06:41:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 3. Install Required Packages

This notebook uses:

```text
transformers
peft
bitsandbytes
accelerate
pandas
gradio
```

In [ ]:
#%pip install -q -U transformers peft bitsandbytes accelerate sentencepiece protobuf pandas gradio
%pip uninstall -y torchao
%pip install -q -U transformers peft accelerate safetensors sentencepiece protobuf pandas gradio

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but

# 4. Import Libraries

In [ ]:
import os
import gc
import re
import shutil
import zipfile
from pathlib import Path

import torch
import pandas as pd
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import PeftModel

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook may run slowly on CPU.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# 5. Main Configuration

Use the same base model that was used during fine-tuning.

## Important

The adapter ZIP is only for upload and download.

```text
Correct PEFT adapter path: /content/lora_qlora_adapter
Wrong PEFT adapter path:   /content/lora_qlora_adapter.zip
```

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

ADAPTER_PATH = "/content/lora_qlora_adapter"
ADAPTER_ZIP_PATH = "/content/lora_qlora_adapter.zip"

USE_4BIT = False

MAX_NEW_TOKENS = 180

SYSTEM_MESSAGE = (
    "You are a helpful AI teacher. "
    "Answer clearly and simply for intermediate students. "
    "If you are unsure, say that you are unsure."
)

print("Base model:", BASE_MODEL)
print("Adapter folder:", ADAPTER_PATH)
print("Adapter zip:", ADAPTER_ZIP_PATH)
print("Use 4-bit:", USE_4BIT)

Base model: Qwen/Qwen2.5-0.5B-Instruct
Adapter folder: /content/lora_qlora_adapter
Adapter zip: /content/lora_qlora_adapter.zip
Use 4-bit: False


# 6. Adapter Folder / ZIP Helper Functions

This section safely handles three cases:

```text
Case 1: Adapter folder already exists in /content/lora_qlora_adapter
Case 2: User uploads lora_qlora_adapter.zip
Case 3: ZIP contains files directly instead of nested folder
```

In [ ]:
def adapter_is_valid(adapter_path=ADAPTER_PATH):
    adapter_dir = Path(adapter_path)

    if not adapter_dir.exists() or not adapter_dir.is_dir():
        return False

    config_file = adapter_dir / "adapter_config.json"

    if not config_file.exists():
        return False

    possible_weight_files = [
        adapter_dir / "adapter_model.safetensors",
        adapter_dir / "adapter_model.bin",
    ]

    return any(path.exists() for path in possible_weight_files)


def show_adapter_files(adapter_path=ADAPTER_PATH):
    adapter_dir = Path(adapter_path)

    print("Adapter folder:", adapter_dir)
    print("Exists:", adapter_dir.exists())
    print("Is folder:", adapter_dir.is_dir())

    if adapter_dir.exists() and adapter_dir.is_dir():
        print("\nFiles:")
        for item in adapter_dir.iterdir():
            print("-", item.name)


def extract_adapter_zip(zip_path=ADAPTER_ZIP_PATH, adapter_path=ADAPTER_PATH):
    zip_path = Path(zip_path)
    adapter_path = Path(adapter_path)

    if not zip_path.exists():
        raise FileNotFoundError(f"Adapter ZIP not found: {zip_path}")

    if adapter_path.exists():
        shutil.rmtree(adapter_path)

    temp_dir = Path("/content/adapter_extract_temp")

    if temp_dir.exists():
        shutil.rmtree(temp_dir)

    temp_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(temp_dir)

    nested_adapter = temp_dir / "lora_qlora_adapter"
    direct_config = temp_dir / "adapter_config.json"

    if nested_adapter.exists() and (nested_adapter / "adapter_config.json").exists():
        shutil.move(str(nested_adapter), str(adapter_path))

    elif direct_config.exists():
        adapter_path.mkdir(parents=True, exist_ok=True)

        for item in temp_dir.iterdir():
            shutil.move(str(item), str(adapter_path / item.name))

    else:
        print("Extracted ZIP contents:")
        for item in temp_dir.rglob("*"):
            print("-", item)

        raise FileNotFoundError(
            "adapter_config.json was not found. "
            "Please upload the correct LoRA / QLoRA adapter ZIP."
        )

    if temp_dir.exists():
        shutil.rmtree(temp_dir, ignore_errors=True)

    if not adapter_is_valid(adapter_path):
        show_adapter_files(adapter_path)
        raise ValueError("Adapter extraction completed, but adapter folder is not valid.")

    print("Adapter extracted and verified successfully.")
    show_adapter_files(adapter_path)


print("Helper functions ready.")

Helper functions ready.


# 7. Upload Adapter ZIP Only If Needed

Run this cell.

If the adapter folder already exists, it will continue without asking for upload.

If not, it will ask you to upload:

```text
lora_qlora_adapter.zip
```

In [ ]:
if adapter_is_valid(ADAPTER_PATH):
    print("Adapter already available.")
    show_adapter_files(ADAPTER_PATH)

else:
    print("Adapter folder not found or incomplete.")
    print("Please upload lora_qlora_adapter.zip")

    from google.colab import files

    uploaded = files.upload()

    if not uploaded:
        raise FileNotFoundError("No adapter ZIP uploaded.")

    uploaded_zip_name = None

    for file_name in uploaded.keys():
        if file_name.lower().endswith(".zip"):
            uploaded_zip_name = file_name
            break

    if uploaded_zip_name is None:
        raise ValueError("Please upload a .zip file containing the LoRA / QLoRA adapter.")

    source_zip = Path(uploaded_zip_name)
    target_zip = Path(ADAPTER_ZIP_PATH)

    # Colab often saves uploaded files directly into /content,
    # so source and target can be the same file.
    if source_zip.resolve() != target_zip.resolve():
        shutil.copy2(source_zip, target_zip)
        print("Uploaded ZIP copied to:", target_zip)
    else:
        print("Uploaded ZIP is already at the correct path:", target_zip)

    extract_adapter_zip(target_zip, ADAPTER_PATH)


Adapter folder not found or incomplete.
Please upload lora_qlora_adapter.zip


Saving lora_qlora_adapter.zip to lora_qlora_adapter.zip
Uploaded ZIP is already at the correct path: /content/lora_qlora_adapter.zip
Adapter extracted and verified successfully.
Adapter folder: /content/lora_qlora_adapter
Exists: True
Is folder: True

Files:
- chat_template.jinja
- tokenizer_config.json
- README.md
- adapter_config.json
- tokenizer.json
- checkpoint-5
- adapter_model.safetensors


# 8. Load Tokenizer

The tokenizer is loaded from the adapter folder if possible.

If tokenizer files are missing from the adapter folder, it loads from the base model.

In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_PATH,
        trust_remote_code=True
    )
    tokenizer_source = ADAPTER_PATH

except Exception as e:
    print("Tokenizer could not be loaded from adapter folder.")
    print("Reason:", e)
    print("Loading tokenizer from base model.")

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL,
        trust_remote_code=True
    )
    tokenizer_source = BASE_MODEL

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded from:", tokenizer_source)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Tokenizer loaded from: /content/lora_qlora_adapter
Pad token: <|endoftext|>
EOS token: <|im_end|>


# 9. Load Base Model

This loads the original base model.

After this, we attach the LoRA / QLoRA adapter.

In [ ]:
def load_base_model():
    """
    Load the base model safely.

    For this evaluation notebook, USE_4BIT=False is recommended because
    Qwen/Qwen2.5-0.5B-Instruct is small enough for Colab GPU/CPU.

    If USE_4BIT=True but bitsandbytes is missing or incompatible,
    this function automatically falls back to normal loading.
    """
    if USE_4BIT:
        try:
            print("Trying 4-bit model loading with bitsandbytes...")

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )

            model = AutoModelForCausalLM.from_pretrained(
                BASE_MODEL,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )

            model.eval()
            print("Base model loaded in 4-bit.")
            return model

        except Exception as e:
            print("4-bit loading failed.")
            print("Reason:", e)
            print("Falling back to normal model loading...")

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

    model.eval()
    print("Base model loaded without 4-bit quantization.")
    return model


base_model = load_base_model()

print("Base model loaded.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded without 4-bit quantization.
Base model loaded.


# 10. Load LoRA / QLoRA Adapter

This creates:

```text
Fine-tuned model = Base model + LoRA / QLoRA adapter
```

To compare fairly without loading two large models, this notebook uses:

```text
fine_tuned_model.disable_adapter()
```

when generating base-model answers.

In [ ]:
if not adapter_is_valid(ADAPTER_PATH):
    raise ValueError("Adapter folder is not valid. Please upload/extract the correct adapter.")

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

fine_tuned_model.eval()

print("Fine-tuned adapter loaded successfully.")
print("Trainable parameters are not needed during evaluation.")

Fine-tuned adapter loaded successfully.
Trainable parameters are not needed during evaluation.


# 11. Generation Functions

Important improvement:

For evaluation, we use:

```python
do_sample = False
```

This makes the comparison deterministic and fair.

Random sampling is useful for creative demos, but not for evaluation.

In [ ]:
def clean_response(text):
    text = str(text)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    return text.strip()


def build_prompt(question):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": str(question)}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def _generate_raw(model_to_use, question, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=0.7):
    prompt = build_prompt(question)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model_to_use.device)

    generation_kwargs = {
        "max_new_tokens": int(max_new_tokens),
        "pad_token_id": tokenizer.eos_token_id,
        "do_sample": bool(do_sample),
    }

    if do_sample:
        generation_kwargs["temperature"] = float(temperature)
        generation_kwargs["top_p"] = 0.9

    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            **generation_kwargs
        )

    response = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return clean_response(response)


def generate_fine_tuned_answer(question, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=0.7):
    return _generate_raw(
        fine_tuned_model,
        question,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature
    )


def generate_base_answer(question, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=0.7):
    if hasattr(fine_tuned_model, "disable_adapter"):
        with fine_tuned_model.disable_adapter():
            return _generate_raw(
                fine_tuned_model,
                question,
                max_new_tokens=max_new_tokens,
                do_sample=do_sample,
                temperature=temperature
            )

    # Fallback
    return _generate_raw(
        base_model,
        question,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature
    )


print("Generation functions ready.")

Generation functions ready.


# 12. Quick Sanity Test

This tests both modes:

```text
Base model answer
Fine-tuned model answer
```

In [ ]:
test_question = "What is LoRA?"

print("Question:", test_question)

print("\nBase model answer:")
print(generate_base_answer(test_question, do_sample=False))

print("\nFine-tuned model answer:")
print(generate_fine_tuned_answer(test_question, do_sample=False))

Question: What is LoRA?

Base model answer:
LoRA stands for Low-Rank Matrix Approximation. It's a type of machine learning algorithm used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (like building blocks) that represent different parts of an image. These blocks are called "low-rank matrices." The LoRA algorithm helps us find the best way to combine these small pieces into one big picture or label, making it easier for computers to understand images.

Fine-tuned model answer:
LoRA stands for Long Short-Term Memory. It's an architecture designed to improve the performance of recurrent neural networks (RNNs) by incorporating long-term dependencies into the model. LoRA helps in capturing longer temporal patterns and reduces overfitting by allowing the network to learn more complex representations at different time steps.


# 13. Test Question Set

You can edit or expand this list.

The questions are designed to check whether the fine-tuned model learned the course-specific explanation style.

In [ ]:
test_questions = [
    "What is fine-tuning?",
    "What is LoRA?",
    "What is QLoRA?",
    "Why is full fine-tuning expensive?",
    "What is the difference between LoRA and QLoRA?",
    "What is the rank r in LoRA?",
    "What is LoRA alpha?",
    "What is LoRA dropout?",
    "What is PEFT?",
    "When should we use QLoRA?",
    "What is quantization?",
    "What is the difference between prompting and fine-tuning?",
]

print("Total questions:", len(test_questions))
for i, q in enumerate(test_questions, start=1):
    print(f"{i}. {q}")

Total questions: 12
1. What is fine-tuning?
2. What is LoRA?
3. What is QLoRA?
4. Why is full fine-tuning expensive?
5. What is the difference between LoRA and QLoRA?
6. What is the rank r in LoRA?
7. What is LoRA alpha?
8. What is LoRA dropout?
9. What is PEFT?
10. When should we use QLoRA?
11. What is quantization?
12. What is the difference between prompting and fine-tuning?


# 14. Generate Base vs Fine-Tuned Comparison

This creates `comparison_df`.

Because we use deterministic generation, comparison results are more stable.

In [ ]:
comparison_results = []

for i, question in enumerate(test_questions, start=1):
    print("=" * 100)
    print(f"Question {i}: {question}")

    base_answer = generate_base_answer(
        question,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False
    )

    fine_tuned_answer = generate_fine_tuned_answer(
        question,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False
    )

    comparison_results.append(
        {
            "question_id": i,
            "question": question,
            "base_model_answer": base_answer,
            "fine_tuned_model_answer": fine_tuned_answer
        }
    )

    print("\nBase model answer:")
    print(base_answer)

    print("\nFine-tuned model answer:")
    print(fine_tuned_answer)

comparison_df = pd.DataFrame(comparison_results)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.width", 1400)

comparison_df

Question 1: What is fine-tuning?

Base model answer:
Fine-tuning is a technique used in machine learning to improve the performance of a model by adjusting its parameters based on specific tasks or datasets. Imagine you have a toy car that needs to learn how to drive safely on different roads. Fine-tuning would be like tweaking the car's wheels and brakes to make it better at navigating various paths and conditions. This process helps the model adapt to new situations more effectively.

Fine-tuned model answer:
Fine-tuning is a technique used in machine learning to improve the performance of a pre-trained model by adjusting its parameters based on specific tasks or datasets. It involves training a new model (the fine-tuned model) with additional data from the task it was trained on, while keeping the original model unchanged. This process helps the fine-tuned model learn more effectively and generalize better across different domains.
Question 2: What is LoRA?

Base model answer:
LoRA 

,question_id,question,base_model_answer,fine_tuned_model_answer
0,1,What is fine-tuning?,Fine-tuning is a technique used in machine learning to improve the performance of a model by adjusting its parameters based on specific tasks or datasets. Imagine you have a toy car that needs to learn how to drive safely on different roads. Fine...,Fine-tuning is a technique used in machine learning to improve the performance of a pre-trained model by adjusting its parameters based on specific tasks or datasets. It involves training a new model (the fine-tuned model) with additional data fr...
1,2,What is LoRA?,LoRA stands for Low-Rank Matrix Approximation. It's a type of machine learning algorithm used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (like building blocks) that represent different parts of an ima...,LoRA stands for Long Short-Term Memory. It's an architecture designed to improve the performance of recurrent neural networks (RNNs) by incorporating long-term dependencies into the model. LoRA helps in capturing longer temporal patterns and redu...
2,3,What is QLoRA?,"QLoRA stands for ""Quantized Low-Rank Approximation."" It's a type of machine learning model used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (called ""quantization units"") that can represent different pa...","QLoRA stands for ""Quantized Low-Rank Approximation."" It's a type of deep learning model optimization technique used in machine learning to improve the efficiency and accuracy of training models by approximating low-rank matrices with quantized va..."
3,4,Why is full fine-tuning expensive?,"Full fine-tuning can be expensive because it involves training the model on a very large dataset, which requires significant computational resources. This process typically takes a lot of time and money to complete. Additionally, fine-tuning mode...",Full fine-tuning involves training the model on a large dataset with additional labeled data to improve its performance. This process can be computationally intensive due to the need to retrain the entire model from scratch. The cost of this appr...
4,5,What is the difference between LoRA and QLoRA?,"LoRA stands for Low-Rank Matrix Approximation, which is used in machine learning to approximate large matrices with smaller ones. It's like when you have a really big puzzle piece but can't fit it all into one spot - LoRA helps by breaking down t...","LoRA stands for Low-Rank Approximation, which is a technique used in deep learning to reduce the computational cost of matrix multiplication by approximating it with lower-rank matrices. QLoRA is an improved version of LoRA designed specifically ..."
5,6,What is the rank r in LoRA?,"In Long Range (LoRA) technology, ""r"" stands for range. It's a measure of how far a signal can travel before fading or becoming unusable. The higher the value of r, the farther the signal can travel without losing quality. A lower r indicates bett...","In Long Short-Term Memory (LSTM) networks, Rank R is typically used as the initial value of the cell state during training. It represents the average of all previous cell states."
6,7,What is LoRA alpha?,"LoRA stands for Low-Rank Approximation. It's a technique used in machine learning to improve the performance of neural networks by reducing their complexity. In simpler terms, it helps make models smaller while still being able to learn from data...","LoRA Alpha refers to the version of LoRA (Local Response Normalization) that includes an additional parameter called ""alpha"". This parameter controls how much weight LoRA assigns to the local response in the network. The default value of alpha is..."
7,8,What is LoRA dropout?,"LoRA stands for Long Short-Term Recurrent Neural Networks. Dropout is a technique used in deep learning to prevent overfitting by randomly dropping out some neurons during training. In LoRA, the same neuron is dropped out multiple times, which 

# 15. Evaluation Rubric

Use the following rubric.

| Criterion | Score | Meaning |
|---|---:|---|
| Accuracy | 1–5 | Is the answer factually correct? |
| Relevance | 1–5 | Does it directly answer the question? |
| Clarity | 1–5 | Is it easy for students to understand? |
| Completeness | 1–5 | Does it include enough useful detail? |
| Hallucination Risk | 1–5 | 1 = low risk, 5 = high risk |
| Teaching Style | 1–5 | Is it aligned with classroom explanation style? |

Important:

```text
For hallucination risk, lower is better.
For all other criteria, higher is better.
```

# 16. Create Manual Evaluation Table Safely

This section fixes the previous issue.

The score columns are created as numeric nullable integer columns:

```text
Int64
```

So this will work:

```python
evaluation_df.loc[1, "base_accuracy"] = 1
```

In [ ]:
if "comparison_df" not in globals():
    raise NameError("comparison_df not found. Run the comparison cell first.")

if comparison_df.empty:
    raise ValueError("comparison_df is empty. Generate base vs fine-tuned answers first.")

required_columns = [
    "question_id",
    "question",
    "base_model_answer",
    "fine_tuned_model_answer",
]

missing_columns = [col for col in required_columns if col not in comparison_df.columns]

if missing_columns:
    raise ValueError(f"comparison_df is missing required columns: {missing_columns}")

evaluation_df = comparison_df.copy()

numeric_score_columns = [
    "base_accuracy",
    "base_relevance",
    "base_clarity",
    "base_completeness",
    "base_hallucination_risk",
    "base_teaching_style",
    "fine_tuned_accuracy",
    "fine_tuned_relevance",
    "fine_tuned_clarity",
    "fine_tuned_completeness",
    "fine_tuned_hallucination_risk",
    "fine_tuned_teaching_style",
]

text_columns = [
    "better_model",
    "reason",
]

for col in numeric_score_columns:
    evaluation_df[col] = pd.Series([pd.NA] * len(evaluation_df), dtype="Int64")

for col in text_columns:
    evaluation_df[col] = ""

print("Manual evaluation table created successfully.")
print("Rows:", len(evaluation_df))
print("Columns:", len(evaluation_df.columns))
print("\nData types:")
print(evaluation_df[numeric_score_columns + text_columns].dtypes)

evaluation_df.head(12)

Manual evaluation table created successfully.
Rows: 12
Columns: 18

Data types:
base_accuracy                    Int64
base_relevance                   Int64
base_clarity                     Int64
base_completeness                Int64
base_hallucination_risk          Int64
base_teaching_style              Int64
fine_tuned_accuracy              Int64
fine_tuned_relevance             Int64
fine_tuned_clarity               Int64
fine_tuned_completeness          Int64
fine_tuned_hallucination_risk    Int64
fine_tuned_teaching_style        Int64
better_model                       str
reason                             str
dtype: object


,question_id,question,base_model_answer,fine_tuned_model_answer,base_accuracy,base_relevance,base_clarity,base_completeness,base_hallucination_risk,base_teaching_style,fine_tuned_accuracy,fine_tuned_relevance,fine_tuned_clarity,fine_tuned_completeness,fine_tuned_hallucination_risk,fine_tuned_teaching_style,better_model,reason
0,1,What is fine-tuning?,Fine-tuning is a technique used in machine learning to improve the performance of a model by adjusting its parameters based on specific tasks or datasets. Imagine you have a toy car that needs to learn how to drive safely on different roads. Fine...,Fine-tuning is a technique used in machine learning to improve the performance of a pre-trained model by adjusting its parameters based on specific tasks or datasets. It involves training a new model (the fine-tuned model) with additional data fr...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
1,2,What is LoRA?,LoRA stands for Low-Rank Matrix Approximation. It's a type of machine learning algorithm used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (like building blocks) that represent different parts of an ima...,LoRA stands for Long Short-Term Memory. It's an architecture designed to improve the performance of recurrent neural networks (RNNs) by incorporating long-term dependencies into the model. LoRA helps in capturing longer temporal patterns and redu...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
2,3,What is QLoRA?,"QLoRA stands for ""Quantized Low-Rank Approximation."" It's a type of machine learning model used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (called ""quantization units"") that can represent different pa...","QLoRA stands for ""Quantized Low-Rank Approximation."" It's a type of deep learning model optimization technique used in machine learning to improve the efficiency and accuracy of training models by approximating low-rank matrices with quantized va...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
3,4,Why is full fine-tuning expensive?,"Full fine-tuning can be expensive because it involves training the model on a very large dataset, which requires significant computational resources. This process typically takes a lot of time and money to complete. Additionally, fine-tuning mode...",Full fine-tuning involves training the model on a large dataset with additional labeled data to improve its performance. This process can be computationally intensive due to the need to retrain the entire model from scratch. The cost of this appr...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
4,5,What is the difference between LoRA and QLoRA?,"LoRA stands for Low-Rank Matrix Approximation, which is used in machine learning to approximate large matrices with smaller ones. It's like when you have a really big puzzle piece but can't fit it all into one spot - LoRA helps by breaking down t...","LoRA stands for Low-Rank Approximation, which is a technique used in deep learning to reduce the computational cost of matrix multiplication by approximating it with lower-rank matrices. QLoRA is an improved version of LoRA designed specifically ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
5,6,What is the rank r in LoRA?,"In Long Range (LoRA) technology, ""r"" stands for range. It's a measure of how far a signal can travel before fading or becoming unusable. The higher the value of r, the farther the signal can travel without losing quality. A lower r indicates bett...","In Long Short-Term Memory (LSTM) networks, Rank R is typically used as the initial value of the cell state during training. It represents the average of all previous cell states.",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
6,7,What is LoRA alpha?,"LoRA stands for Low-Rank Approximation. It's a technique used in machine learning to improve the performance of neural networks by reducing 

# 17. Example Manual Scoring

This cell shows how to fill scores safely.

You can edit the row index and scores based on actual answers.

In [ ]:
# Example scoring for row 0
# Change these scores according to your actual model answers.

row_index = 0

evaluation_df.loc[row_index, "base_accuracy"] = 3
evaluation_df.loc[row_index, "base_relevance"] = 4
evaluation_df.loc[row_index, "base_clarity"] = 3
evaluation_df.loc[row_index, "base_completeness"] = 3
evaluation_df.loc[row_index, "base_hallucination_risk"] = 2
evaluation_df.loc[row_index, "base_teaching_style"] = 3

evaluation_df.loc[row_index, "fine_tuned_accuracy"] = 4
evaluation_df.loc[row_index, "fine_tuned_relevance"] = 5
evaluation_df.loc[row_index, "fine_tuned_clarity"] = 5
evaluation_df.loc[row_index, "fine_tuned_completeness"] = 4
evaluation_df.loc[row_index, "fine_tuned_hallucination_risk"] = 1
evaluation_df.loc[row_index, "fine_tuned_teaching_style"] = 5

evaluation_df.loc[row_index, "better_model"] = "Fine-tuned model"
evaluation_df.loc[row_index, "reason"] = "The fine-tuned answer is clearer and more aligned with classroom teaching style."

evaluation_df.loc[[row_index]]

,question_id,question,base_model_answer,fine_tuned_model_answer,base_accuracy,base_relevance,base_clarity,base_completeness,base_hallucination_risk,base_teaching_style,fine_tuned_accuracy,fine_tuned_relevance,fine_tuned_clarity,fine_tuned_completeness,fine_tuned_hallucination_risk,fine_tuned_teaching_style,better_model,reason
0,1,What is fine-tuning?,Fine-tuning is a technique used in machine learning to improve the performance of a model by adjusting its parameters based on specific tasks or datasets. Imagine you have a toy car that needs to learn how to drive safely on different roads. Fine...,Fine-tuning is a technique used in machine learning to improve the performance of a pre-trained model by adjusting its parameters based on specific tasks or datasets. It involves training a new model (the fine-tuned model) with additional data fr...,3,4,3,3,2,3,4,5,5,4,1,5,Fine-tuned model,The fine-tuned answer is clearer and more aligned with classroom teaching style.


# 18. Optional: Auto-Suggest Scores Using Keyword Rules

This is not a replacement for human evaluation.

It only gives a quick starting point by checking whether the answer contains important keywords.

In [ ]:
expected_keywords = {
    "What is fine-tuning?": ["pre-trained", "train", "specific", "task"],
    "What is LoRA?": ["low-rank", "adaptation", "adapter"],
    "What is QLoRA?": ["quantized", "lora", "4-bit"],
    "Why is full fine-tuning expensive?": ["parameters", "memory", "time", "cost"],
    "What is the difference between LoRA and QLoRA?": ["4-bit", "quantized", "adapter"],
    "What is the rank r in LoRA?": ["rank", "matrix", "capacity"],
    "What is LoRA alpha?": ["scaling", "factor"],
    "What is LoRA dropout?": ["dropout", "overfitting"],
    "What is PEFT?": ["parameter-efficient", "fine-tuning"],
    "When should we use QLoRA?": ["memory", "limited", "4-bit"],
    "What is quantization?": ["precision", "4-bit", "memory"],
    "What is the difference between prompting and fine-tuning?": ["prompt", "training", "behavior"],
}


def keyword_score(answer, keywords):
    answer_lower = str(answer).lower()
    hits = sum(1 for kw in keywords if kw.lower() in answer_lower)

    if hits >= 3:
        return 5
    if hits == 2:
        return 4
    if hits == 1:
        return 3
    return 1


def auto_suggest_scores(df):
    suggested = df.copy()

    for idx, row in suggested.iterrows():
        question = row["question"]
        keywords = expected_keywords.get(question, [])

        if not keywords:
            continue

        base_k_score = keyword_score(row["base_model_answer"], keywords)
        fine_k_score = keyword_score(row["fine_tuned_model_answer"], keywords)

        suggested.loc[idx, "base_accuracy"] = base_k_score
        suggested.loc[idx, "base_relevance"] = min(5, base_k_score + 1)
        suggested.loc[idx, "base_clarity"] = 3
        suggested.loc[idx, "base_completeness"] = base_k_score
        suggested.loc[idx, "base_hallucination_risk"] = 6 - min(5, base_k_score)
        suggested.loc[idx, "base_teaching_style"] = 3

        suggested.loc[idx, "fine_tuned_accuracy"] = fine_k_score
        suggested.loc[idx, "fine_tuned_relevance"] = min(5, fine_k_score + 1)
        suggested.loc[idx, "fine_tuned_clarity"] = 4
        suggested.loc[idx, "fine_tuned_completeness"] = fine_k_score
        suggested.loc[idx, "fine_tuned_hallucination_risk"] = 6 - min(5, fine_k_score)
        suggested.loc[idx, "fine_tuned_teaching_style"] = 4

        if fine_k_score > base_k_score:
            suggested.loc[idx, "better_model"] = "Fine-tuned model"
            suggested.loc[idx, "reason"] = "Fine-tuned answer contains more expected concepts."
        elif base_k_score > fine_k_score:
            suggested.loc[idx, "better_model"] = "Base model"
            suggested.loc[idx, "reason"] = "Base answer contains more expected concepts."
        else:
            suggested.loc[idx, "better_model"] = "Tie / Human review needed"
            suggested.loc[idx, "reason"] = "Both answers need human review."

    return suggested


# Uncomment this line if you want automatic suggested scores:
# evaluation_df = auto_suggest_scores(evaluation_df)

evaluation_df.head()

,question_id,question,base_model_answer,fine_tuned_model_answer,base_accuracy,base_relevance,base_clarity,base_completeness,base_hallucination_risk,base_teaching_style,fine_tuned_accuracy,fine_tuned_relevance,fine_tuned_clarity,fine_tuned_completeness,fine_tuned_hallucination_risk,fine_tuned_teaching_style,better_model,reason
0,1,What is fine-tuning?,Fine-tuning is a technique used in machine learning to improve the performance of a model by adjusting its parameters based on specific tasks or datasets. Imagine you have a toy car that needs to learn how to drive safely on different roads. Fine...,Fine-tuning is a technique used in machine learning to improve the performance of a pre-trained model by adjusting its parameters based on specific tasks or datasets. It involves training a new model (the fine-tuned model) with additional data fr...,3,4,3,3,2,3,4,5,5,4,1,5,Fine-tuned model,The fine-tuned answer is clearer and more aligned with classroom teaching style.
1,2,What is LoRA?,LoRA stands for Low-Rank Matrix Approximation. It's a type of machine learning algorithm used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (like building blocks) that represent different parts of an ima...,LoRA stands for Long Short-Term Memory. It's an architecture designed to improve the performance of recurrent neural networks (RNNs) by incorporating long-term dependencies into the model. LoRA helps in capturing longer temporal patterns and redu...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
2,3,What is QLoRA?,"QLoRA stands for ""Quantized Low-Rank Approximation."" It's a type of machine learning model used in computer vision tasks like image recognition. Imagine you have a bunch of tiny pieces (called ""quantization units"") that can represent different pa...","QLoRA stands for ""Quantized Low-Rank Approximation."" It's a type of deep learning model optimization technique used in machine learning to improve the efficiency and accuracy of training models by approximating low-rank matrices with quantized va...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
3,4,Why is full fine-tuning expensive?,"Full fine-tuning can be expensive because it involves training the model on a very large dataset, which requires significant computational resources. This process typically takes a lot of time and money to complete. Additionally, fine-tuning mode...",Full fine-tuning involves training the model on a large dataset with additional labeled data to improve its performance. This process can be computationally intensive due to the need to retrain the entire model from scratch. The cost of this appr...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,
4,5,What is the difference between LoRA and QLoRA?,"LoRA stands for Low-Rank Matrix Approximation, which is used in machine learning to approximate large matrices with smaller ones. It's like when you have a really big puzzle piece but can't fit it all into one spot - LoRA helps by breaking down t...","LoRA stands for Low-Rank Approximation, which is a technique used in deep learning to reduce the computational cost of matrix multiplication by approximating it with lower-rank matrices. QLoRA is an improved version of LoRA designed specifically ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,,


# 19. Calculate Total Scores

Formula:

```text
Total Score = accuracy + relevance + clarity + completeness + teaching_style + converted_hallucination_score
```

Hallucination conversion:

```text
risk 1 → score 5
risk 2 → score 4
risk 3 → score 3
risk 4 → score 2
risk 5 → score 1
```

In [ ]:
def safe_int(value):
    try:
        if pd.isna(value):
            return 0
        return int(value)
    except Exception:
        return 0


def converted_hallucination_score(value):
    risk = safe_int(value)

    if risk == 0:
        return 0

    return max(0, 6 - risk)


def calculate_total_scores(row):
    base_total = (
        safe_int(row["base_accuracy"])
        + safe_int(row["base_relevance"])
        + safe_int(row["base_clarity"])
        + safe_int(row["base_completeness"])
        + safe_int(row["base_teaching_style"])
        + converted_hallucination_score(row["base_hallucination_risk"])
    )

    fine_total = (
        safe_int(row["fine_tuned_accuracy"])
        + safe_int(row["fine_tuned_relevance"])
        + safe_int(row["fine_tuned_clarity"])
        + safe_int(row["fine_tuned_completeness"])
        + safe_int(row["fine_tuned_teaching_style"])
        + converted_hallucination_score(row["fine_tuned_hallucination_risk"])
    )

    return pd.Series(
        {
            "base_total_score": base_total,
            "fine_tuned_total_score": fine_total,
            "score_difference": fine_total - base_total
        }
    )


for col in ["base_total_score", "fine_tuned_total_score", "score_difference"]:
    if col in evaluation_df.columns:
        evaluation_df = evaluation_df.drop(columns=[col])

score_df = evaluation_df.apply(calculate_total_scores, axis=1)

evaluation_df = pd.concat([evaluation_df, score_df], axis=1)

evaluation_df[
    [
        "question_id",
        "question",
        "base_total_score",
        "fine_tuned_total_score",
        "score_difference",
        "better_model",
        "reason"
    ]
]

,question_id,question,base_total_score,fine_tuned_total_score,score_difference,better_model,reason
0,1,What is fine-tuning?,20,28,8,Fine-tuned model,The fine-tuned answer is clearer and more aligned with classroom teaching style.
1,2,What is LoRA?,0,0,0,,
2,3,What is QLoRA?,0,0,0,,
3,4,Why is full fine-tuning expensive?,0,0,0,,
4,5,What is the difference between LoRA and QLoRA?,0,0,0,,
5,6,What is the rank r in LoRA?,0,0,0,,
6,7,What is LoRA alpha?,0,0,0,,
7,8,What is LoRA dropout?,0,0,0,,
8,9,What is PEFT?,0,0,0,,
9,10,When should we use QLoRA?,0,0,0,,


# 20. Evaluation Dashboard

This summarizes whether the fine-tuned model improved.

Scores are meaningful after you fill the evaluation table.

In [ ]:
def create_evaluation_summary(df):
    total_questions = len(df)

    base_avg = df["base_total_score"].mean()
    fine_avg = df["fine_tuned_total_score"].mean()

    improved_rows = (df["fine_tuned_total_score"] > df["base_total_score"]).sum()
    declined_rows = (df["fine_tuned_total_score"] < df["base_total_score"]).sum()
    tied_rows = (df["fine_tuned_total_score"] == df["base_total_score"]).sum()

    summary = pd.DataFrame(
        [
            ["Total Questions", total_questions],
            ["Average Base Score", round(base_avg, 2)],
            ["Average Fine-Tuned Score", round(fine_avg, 2)],
            ["Questions Improved", improved_rows],
            ["Questions Declined", declined_rows],
            ["Questions Tied", tied_rows],
        ],
        columns=["Metric", "Value"]
    )

    return summary


evaluation_summary_df = create_evaluation_summary(evaluation_df)
evaluation_summary_df

,Metric,Value
0,Total Questions,12.00
1,Average Base Score,1.67
2,Average Fine-Tuned Score,2.33
3,Questions Improved,1.00
4,Questions Declined,0.00
5,Questions Tied,11.00


# 21. Export Reports

This exports:

```text
base_vs_finetuned_comparison.csv
fine_tuned_model_evaluation.csv
evaluation_summary.csv
```

In [ ]:
comparison_csv_path = "/content/base_vs_finetuned_comparison.csv"
evaluation_csv_path = "/content/fine_tuned_model_evaluation.csv"
summary_csv_path = "/content/evaluation_summary.csv"

comparison_df.to_csv(comparison_csv_path, index=False)
evaluation_df.to_csv(evaluation_csv_path, index=False)
evaluation_summary_df.to_csv(summary_csv_path, index=False)

print("Saved:", comparison_csv_path)
print("Saved:", evaluation_csv_path)
print("Saved:", summary_csv_path)

Saved: /content/base_vs_finetuned_comparison.csv
Saved: /content/fine_tuned_model_evaluation.csv
Saved: /content/evaluation_summary.csv


In [ ]:
from google.colab import files

files.download(comparison_csv_path)
files.download(evaluation_csv_path)
files.download(summary_csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 22. Improved Gradio App

This UI has three useful modes:

```text
1. Fine-tuned assistant
2. Base vs fine-tuned comparison
3. Evaluation guide
```

In [ ]:
def gradio_finetuned_answer(question, temperature, max_tokens, use_sampling):
    if not question or not str(question).strip():
        return "Please enter a question."

    try:
        return generate_fine_tuned_answer(
            question,
            max_new_tokens=int(max_tokens),
            do_sample=bool(use_sampling),
            temperature=float(temperature)
        )
    except Exception as e:
        return f"Error: {str(e)}"


def gradio_compare_answers(question, max_tokens):
    if not question or not str(question).strip():
        return "Please enter a question.", "Please enter a question.", "No comparison yet."

    try:
        base_answer = generate_base_answer(
            question,
            max_new_tokens=int(max_tokens),
            do_sample=False
        )

        fine_answer = generate_fine_tuned_answer(
            question,
            max_new_tokens=int(max_tokens),
            do_sample=False
        )

        comparison_note = (
            "Read both answers and judge using the rubric: "
            "accuracy, relevance, clarity, completeness, hallucination risk, and teaching style."
        )

        return base_answer, fine_answer, comparison_note

    except Exception as e:
        error_text = f"Error: {str(e)}"
        return error_text, error_text, error_text


custom_css = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    font-family: 'Inter', 'Segoe UI', sans-serif;
}

#hero {
    padding: 34px;
    border-radius: 30px;
    background:
        radial-gradient(circle at top left, rgba(59,130,246,.42), transparent 28%),
        radial-gradient(circle at top right, rgba(168,85,247,.35), transparent 30%),
        linear-gradient(135deg, #020617, #0f172a 56%, #111827);
    color: white;
    margin-bottom: 22px;
    border: 1px solid rgba(148,163,184,.30);
    box-shadow: 0 24px 70px rgba(15,23,42,.35);
}

#hero h1 {
    font-size: 42px;
    margin-bottom: 8px;
    background: linear-gradient(90deg, #38bdf8, #a78bfa, #22c55e);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

#hero p {
    color: #dbeafe;
    font-size: 16px;
    line-height: 1.6;
}

.info-card {
    padding: 18px;
    border-radius: 18px;
    background: linear-gradient(135deg, #f8fafc, #e2e8f0);
    border: 1px solid #cbd5e1;
    min-height: 130px;
}

.info-card h3 {
    color: #0f172a;
    margin-bottom: 8px;
}

.info-card p {
    color: #334155;
    font-size: 14px;
}
"""

with gr.Blocks(
    css=custom_css,
    title="Fine-Tuned LLM Evaluation Studio",
    theme=gr.themes.Soft()
) as demo:

    gr.HTML(
        """
        <div id="hero">
            <h1>Fine-Tuned LLM Evaluation Studio</h1>
            <p>
                Test your LoRA / QLoRA adapter, compare base vs fine-tuned answers,
                and prepare classroom evaluation evidence.
            </p>
            <p>
                <b>Workflow:</b> Load Adapter → Ask Questions → Compare Answers → Evaluate → Export Results
            </p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column():
            gr.HTML(
                """
                <div class="info-card">
                    <h3>Adapter Model</h3>
                    <p>Uses base model plus your saved LoRA / QLoRA adapter.</p>
                    <p><b>Path:</b> /content/lora_qlora_adapter</p>
                </div>
                """
            )
        with gr.Column():
            gr.HTML(
                """
                <div class="info-card">
                    <h3>Fair Evaluation</h3>
                    <p>Comparison mode uses deterministic generation for stable results.</p>
                    <p><b>Setting:</b> do_sample=False</p>
                </div>
                """
            )
        with gr.Column():
            gr.HTML(
                """
                <div class="info-card">
                    <h3>Evaluation Rubric</h3>
                    <p>Score accuracy, relevance, clarity, completeness, hallucination risk, and teaching style.</p>
                </div>
                """
            )

    with gr.Tabs():

        with gr.Tab("Fine-Tuned Assistant"):
            gr.Markdown("### Ask your fine-tuned model")

            with gr.Row():
                with gr.Column(scale=1):
                    ft_question = gr.Textbox(
                        label="Question",
                        placeholder="Example: Explain LoRA in simple words.",
                        lines=5
                    )

                    ft_sampling = gr.Checkbox(
                        label="Use sampling for creative answer",
                        value=False
                    )

                    ft_temperature = gr.Slider(
                        minimum=0.1,
                        maximum=1.0,
                        value=0.7,
                        step=0.1,
                        label="Temperature"
                    )

                    ft_tokens = gr.Slider(
                        minimum=50,
                        maximum=400,
                        value=180,
                        step=10,
                        label="Max New Tokens"
                    )

                    ft_button = gr.Button("Generate Fine-Tuned Answer", variant="primary")

                with gr.Column(scale=1):
                    ft_output = gr.Textbox(
                        label="Fine-Tuned Model Answer",
                        lines=16
                    )

            ft_button.click(
                fn=gradio_finetuned_answer,
                inputs=[ft_question, ft_temperature, ft_tokens, ft_sampling],
                outputs=ft_output
            )

            ft_question.submit(
                fn=gradio_finetuned_answer,
                inputs=[ft_question, ft_temperature, ft_tokens, ft_sampling],
                outputs=ft_output
            )

        with gr.Tab("Base vs Fine-Tuned Comparison"):
            gr.Markdown("### Compare both models on the same question")

            compare_question = gr.Textbox(
                label="Comparison Question",
                placeholder="Example: What is QLoRA?",
                lines=4
            )

            compare_tokens = gr.Slider(
                minimum=50,
                maximum=400,
                value=180,
                step=10,
                label="Max New Tokens"
            )

            compare_button = gr.Button("Compare Answers", variant="primary")

            with gr.Row():
                base_output = gr.Textbox(
                    label="Base Model Answer",
                    lines=14
                )

                fine_output = gr.Textbox(
                    label="Fine-Tuned Model Answer",
                    lines=14
                )

            compare_note = gr.Textbox(
                label="Evaluation Note",
                lines=3
            )

            compare_button.click(
                fn=gradio_compare_answers,
                inputs=[compare_question, compare_tokens],
                outputs=[base_output, fine_output, compare_note]
            )

        with gr.Tab("Evaluation Guide"):
            gr.Markdown(
                """
                ## Manual Evaluation Rubric

                | Criterion | Score | Meaning |
                |---|---:|---|
                | Accuracy | 1–5 | Is the answer factually correct? |
                | Relevance | 1–5 | Does it directly answer the question? |
                | Clarity | 1–5 | Is it easy for students to understand? |
                | Completeness | 1–5 | Does it include enough useful detail? |
                | Hallucination Risk | 1–5 | 1 = low risk, 5 = high risk |
                | Teaching Style | 1–5 | Is it aligned with classroom explanation style? |

                ## Important

                Fine-tuning is successful only if the fine-tuned model is more accurate,
                more relevant, clearer, and less likely to hallucinate.
                """
            )

demo.launch(share=True)

/tmp/ipykernel_1090/1775348696.py:98: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f6ccfbb3d141bd4935.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 23. Student Lab Task

## Scenario

You trained a LoRA / QLoRA adapter for a Generative AI teaching assistant.

Now your job is to prove whether the fine-tuned model is better than the base model.

## Required Work

```text
1. Upload adapter ZIP
2. Load base model
3. Load adapter
4. Ask at least 12 questions
5. Generate base and fine-tuned answers
6. Fill evaluation scores
7. Export CSV reports
8. Launch Gradio UI
9. Submit screenshots and conclusion
```

## Submission

```text
1. Completed notebook
2. base_vs_finetuned_comparison.csv
3. fine_tuned_model_evaluation.csv
4. evaluation_summary.csv
5. Screenshot of Gradio UI
6. Short conclusion
```

# Final Summary

This regenerated notebook fixes the evaluation-table issue and improves the model testing flow.

## Key improvements

```text
1. Numeric score columns prevent Pandas dtype errors.
2. Deterministic generation makes comparison stable.
3. Adapter ZIP handling is safer.
4. Base model comparison uses adapter disabling to save GPU memory.
5. Gradio UI is more professional and useful for classroom demos.
```

## Main lesson

```text
A fine-tuned model is not automatically better.
It must be compared, scored, evaluated, and tested before deployment.
```